# Tier 4 Solutions — Set 2: Strings and Trees

**Modules covered**: String Algorithms (KMP, Rabin-Karp), Tries, Suffix Arrays, Suffix Trees

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook builds algorithmic thinking used to reason about performance and correctness in computational biology tools.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Big-O is asymptotic guidance; constant factors still matter in practice.
- Correctness comes before optimization: verify edge cases before performance tuning.
- Data structure choice often dominates speed more than micro-optimizations.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


## Problem 1: Naive Pattern Matching (1 star)

In [ ]:
def naive_search(text: str, pattern: str) -> tuple[list[int], int]:
    n, m = len(text), len(pattern)
    positions = []
    comparisons = 0
    for i in range(n - m + 1):
        for j in range(m):
            comparisons += 1
            if text[i + j] != pattern[j]:
                break
        else:
            positions.append(i)
    return positions, comparisons


dna = "AAGAATTCGAATTCTTGCGAATTCATG"
site = "GAATTC"

positions, comparisons = naive_search(dna, site)
print(f"Pattern  : {site}")
print(f"Occurrences (0-indexed): {positions}")
print(f"Total comparisons: {comparisons}")

## Problem 2: KMP Failure Function (1 star)

In [ ]:
def kmp_failure(pattern: str) -> list[int]:
    m = len(pattern)
    failure = [0] * m
    j = 0  # length of previous longest prefix-suffix
    i = 1
    while i < m:
        if pattern[i] == pattern[j]:
            j += 1
            failure[i] = j
            i += 1
        elif j > 0:
            j = failure[j - 1]   # fall back without incrementing i
        else:
            failure[i] = 0
            i += 1
    return failure


pattern = "AABAABAAC"
failure = kmp_failure(pattern)

print(f"Pattern : {' '.join(pattern)}")
print(f"Index   : {' '.join(str(i) for i in range(len(pattern)))}")
print(f"Failure : {' '.join(str(v) for v in failure)}")
print()
# Explanation of selected values:
# failure[0] = 0  — a single character has no proper prefix-suffix
# failure[1] = 1  — "AA" has prefix "A" = suffix "A" of length 1
# failure[2] = 0  — "AAB" has no non-trivial prefix that is also a suffix
# failure[4] = 2  — "AABAA" has prefix "AA" = suffix "AA" of length 2
# failure[7] = 5  — "AABAABAА" has prefix "AABAAB" — wait, let's print them all
for i, v in enumerate(failure):
    print(f"  failure[{i}] = {v}  (pattern[0:{i+1}] = '{pattern[:i+1]}', longest prefix-suffix length = {v})")

## Problem 3: Multi-Pattern Search (2 stars)

In [ ]:
def kmp_search(text: str, pattern: str) -> list[int]:
    if not pattern:
        return []
    failure = kmp_failure(pattern)
    positions = []
    j = 0  # characters matched so far
    for i, ch in enumerate(text):
        while j > 0 and ch != pattern[j]:
            j = failure[j - 1]
        if ch == pattern[j]:
            j += 1
        if j == len(pattern):
            positions.append(i - j + 1)
            j = failure[j - 1]
    return positions


dna = (
    "ACGTACGTGAATTCACGTACGTGGATCCACGTACGT"
    "ACGTACGTGAATTCACGTACGTACGTACGTACGTAC"
)

enzymes = {
    "EcoRI":   "GAATTC",
    "BamHI":   "GGATCC",
    "HindIII": "AAGCTT",
    "NotI":    "GCGGCCGC",
}

for name, site in enzymes.items():
    positions = kmp_search(dna, site)
    print(f"{name:<10}({site}): positions={positions}  count={len(positions)}")

## Problem 4: Trie Autocomplete (2 stars)

In [ ]:
class TrieNode:
    def __init__(self):
        self.children: dict[str, "TrieNode"] = {}
        self.is_end: bool = False


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def autocomplete(self, prefix: str) -> list[str]:
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return []
            node = node.children[ch]
        # DFS from the node at end of prefix
        results = []
        self._dfs(node, prefix, results)
        return sorted(results)

    def _dfs(self, node: TrieNode, current: str, results: list) -> None:
        if node.is_end:
            results.append(current)
        for ch, child in node.children.items():
            self._dfs(child, current + ch, results)


gene_names = [
    "TP53", "TP63", "TP73",
    "BRCA1", "BRCA2",
    "KRAS", "NRAS", "HRAS",
    "EGFR", "ERBB2", "ERBB3",
    "MYC", "MYCN",
    "PTEN", "PIK3CA", "PIK3CB",
    "ATM", "ATR",
    "CDK2", "CDK4", "CDK6",
]

trie = Trie()
for name in gene_names:
    trie.insert(name)

for prefix in ["TP", "BRC", "ERB", "CDK", "XYZ"]:
    results = trie.autocomplete(prefix)
    print(f"autocomplete({prefix!r:6}) -> {results}")

## Problem 5: Suffix Array Construction (2 stars)

In [ ]:
def build_suffix_array(text: str) -> list[int]:
    suffixes = sorted(range(len(text)), key=lambda i: text[i:])
    return suffixes


def verify_suffix_array(text: str, sa: list[int]) -> bool:
    for i in range(len(sa) - 1):
        if text[sa[i]:] >= text[sa[i + 1]:]:
            return False
    return True


text = "BANANA"
sa = build_suffix_array(text)
print(f"Text: {text}")
print(f"Suffix array: {sa}")
print(f"Sorted suffixes:")
for idx in sa:
    print(f"  [{idx}] {text[idx:]}")
print(f"Valid: {verify_suffix_array(text, sa)}")

print()
dna = "ATCGATCGATCGTAGCTAGC"
sa_dna = build_suffix_array(dna)
print(f"DNA suffix array valid: {verify_suffix_array(dna, sa_dna)}")

## Problem 6: Longest Repeated Substring (3 stars)

In [ ]:
def build_lcp_array(text: str, sa: list[int]) -> list[int]:
    n = len(text)
    # rank[i] = position of suffix starting at i in the suffix array
    rank = [0] * n
    for i, s in enumerate(sa):
        rank[s] = i

    lcp = [0] * n
    h = 0
    for i in range(n):
        if rank[i] > 0:
            j = sa[rank[i] - 1]   # previous suffix in sorted order
            while i + h < n and j + h < n and text[i + h] == text[j + h]:
                h += 1
            lcp[rank[i]] = h
            if h > 0:
                h -= 1
    return lcp


def longest_repeated_substring(text: str) -> str:
    sa = build_suffix_array(text)
    lcp = build_lcp_array(text, sa)
    max_len = max(lcp)
    if max_len == 0:
        return ""
    idx = lcp.index(max_len)
    return text[sa[idx]: sa[idx] + max_len]


dna = "ATCGATCGATCGTAGCATCGATCG"
lrs = longest_repeated_substring(dna)
print(f"DNA sequence : {dna}")
print(f"Longest repeated substring: {lrs!r}  (length {len(lrs)})")

count = sum(1 for i in range(len(dna) - len(lrs) + 1) if dna[i:i+len(lrs)] == lrs)
print(f"Occurrences  : {count}")

## Problem 7: Burrows-Wheeler Transform (3 stars)

In [ ]:
def bwt(text: str) -> str:
    if not text.endswith("$"):
        text = text + "$"
    n = len(text)
    rotations = sorted(text[i:] + text[:i] for i in range(n))
    return "".join(r[-1] for r in rotations)


def ibwt(bwt_string: str) -> str:
    n = len(bwt_string)
    # Build (bwt_char, index) pairs and sort to get first column
    table = sorted(range(n), key=lambda i: bwt_string[i])
    # LF-mapping: table[i] = which row in the sorted matrix has bwt_string[i] in F column
    # Reconstruct by following LF-mapping from the row containing '$'
    dollar_row = bwt_string.index("$")
    result = []
    row = dollar_row
    for _ in range(n):
        result.append(bwt_string[row])
        row = table[row]
    # result is the original text in reverse order (we read last column backward)
    return "".join(reversed(result))


for text in ["BANANA", "ATCGATCG", "GATTACA"]:
    transformed = bwt(text)
    recovered = ibwt(transformed)
    original_with_sentinel = text + "$"
    ok = recovered == original_with_sentinel
    print(f"Original : {text}")
    print(f"BWT      : {transformed}")
    print(f"Recovered: {recovered}  {'OK' if ok else 'MISMATCH'}")
    print()

## Problem 8: Aho-Corasick Multi-Pattern Search (3 stars)

In [ ]:
from collections import deque


class AhoCorasick:
    def __init__(self):
        self.goto: list[dict[str, int]] = [{}]
        self.fail: list[int] = [0]
        self.output: list[list[str]] = [[]]

    def add_pattern(self, pattern: str, name: str) -> None:
        state = 0
        for ch in pattern:
            if ch not in self.goto[state]:
                self.goto.append({})
                self.fail.append(0)
                self.output.append([])
                self.goto[state][ch] = len(self.goto) - 1
            state = self.goto[state][ch]
        self.output[state].append(name)

    def build(self) -> None:
        queue = deque()
        # All depth-1 states have fail link -> root
        for ch, s in self.goto[0].items():
            self.fail[s] = 0
            queue.append(s)

        while queue:
            r = queue.popleft()
            for ch, s in self.goto[r].items():
                queue.append(s)
                state = self.fail[r]
                while state != 0 and ch not in self.goto[state]:
                    state = self.fail[state]
                self.fail[s] = self.goto[state].get(ch, 0)
                if self.fail[s] == s:
                    self.fail[s] = 0
                # Merge output of fail state
                self.output[s] = self.output[s] + self.output[self.fail[s]]

    def search(self, text: str) -> list[tuple[int, str]]:
        state = 0
        results = []
        for i, ch in enumerate(text):
            while state != 0 and ch not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(ch, 0)
            for name in self.output[state]:
                # position is the start of the match
                # we need to know each pattern's length — store it during add_pattern
                results.append((i, name))
        return results


# We need pattern lengths to compute start positions correctly.
# Extend AhoCorasick to store pattern lengths:
class AhoCorasickWithPos(AhoCorasick):
    def __init__(self):
        super().__init__()
        self._pattern_len: dict[str, int] = {}

    def add_pattern(self, pattern: str, name: str) -> None:
        super().add_pattern(pattern, name)
        self._pattern_len[name] = len(pattern)

    def search(self, text: str) -> list[tuple[int, str]]:
        state = 0
        results = []
        for i, ch in enumerate(text):
            while state != 0 and ch not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(ch, 0)
            for name in self.output[state]:
                start = i - self._pattern_len[name] + 1
                results.append((start, name))
        return sorted(results)


genome = (
    "ACGTACGTGAATTCACGTACGTGGATCCACGTACGT"
    "ACGTACGTGAATTCACGTACGTACGTACGTACGTAC"
)

enzymes = {
    "EcoRI":   "GAATTC",
    "BamHI":   "GGATCC",
    "HindIII": "AAGCTT",
}

ac = AhoCorasickWithPos()
for name, site in enzymes.items():
    ac.add_pattern(site, name)
ac.build()

matches = ac.search(genome)
for pos, name in matches:
    print(f"{name:<10} at position {pos}")